# Flower Image Classifier — Quickstart (Colab)

Run the trained model on your own flower photo, loaded straight from the Hugging Face Hub. No training needed.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/delcenjo/flower-image-classifier/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip -q install torch torchvision huggingface_hub pillow

In [ ]:
import torch
from torchvision import models, transforms
from huggingface_hub import hf_hub_download
from PIL import Image

ckpt = torch.load(hf_hub_download('delcenjo/flower-image-classifier', 'flower_classifier.pt'), map_location='cpu')
classes = ckpt['classes']
model = models.resnet18(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, len(classes))
model.load_state_dict(ckpt['model_state']); model.eval()

preprocess = transforms.Compose([
    transforms.Resize((128, 128)), transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])

def predict(img):
    x = preprocess(img.convert('RGB')).unsqueeze(0)
    with torch.no_grad():
        probs = model(x).softmax(dim=1)[0]
    return sorted(zip(classes, probs.tolist()), key=lambda t: -t[1])

## Upload a flower photo and classify it

In [ ]:
from google.colab import files
from PIL import Image
uploaded = files.upload()
img = Image.open(next(iter(uploaded)))
for cls, p in predict(img):
    print(f'{cls:12s} {p*100:5.1f}%')